### Cell 1: Setup & Dependencies

In [1]:
# Install required packages for NIfTI processing and deep learning
!pip install -q nibabel scipy torch tqdm --no-deps

import glob
import os
import shutil
import tarfile
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndimage
import torch
import torch.nn as nn
from tqdm import tqdm

print("Environment setup complete. Nibabel version:", nib.__version__)

Environment setup complete. Nibabel version: 5.4.2


---

### Cell 2: Utility Functions & Loss Definitions

In [2]:
def preprocess_mri_volume(volume_data, lower_percentile=1.0, upper_percentile=99.0):
    """
    1. Winsorizes (clips) extreme brightness values to clear scanning artifacts.
    2. Performs Z-score normalization restricted strictly to non-zero brain tissue.
    """
    brain_mask = volume_data > 0
    if not brain_mask.any():
        return volume_data.astype(np.float32)

    # 1. Clip extreme percentiles inside the brain tissue
    brain_voxels = volume_data[brain_mask]
    low_val, high_val = np.percentile(brain_voxels, [lower_percentile, upper_percentile])
    clipped_data = np.clip(volume_data, low_val, high_val)

    # 2. Z-Score normalization on brain tissue only (mean=0, std=1)
    mean_val = np.mean(clipped_data[brain_mask])
    std_val = np.std(clipped_data[brain_mask])

    if std_val < 1e-8:
        std_val = 1.0

    normalized_data = np.zeros_like(volume_data, dtype=np.float32)
    normalized_data[brain_mask] = (clipped_data[brain_mask] - mean_val) / std_val

    return normalized_data


def extract_clinical_metrics(segmentation_mask, voxel_volume_mm3=1.0, min_foci_voxels=500):
    """
    1. Removes tiny noise fragments using 3D connected component labeling
       to rectify inflated multifocality counts.
    2. Calculates true solid tumor volume by excluding Edema (Label 2).
    """
    binary_tumor_mask = (segmentation_mask > 0).astype(np.uint8)
    labeled_array, num_features = ndimage.label(binary_tumor_mask)

    true_foci_count = 0
    cleaned_mask = np.zeros_like(segmentation_mask)

    for cluster_id in range(1, num_features + 1):
        cluster_voxels = (labeled_array == cluster_id)
        if np.sum(cluster_voxels) >= min_foci_voxels:
            true_foci_count += 1
            cleaned_mask[cluster_voxels] = segmentation_mask[cluster_voxels]

    # BraTS Labels: 1 = Necrotic Core, 2 = Edema (Swelling), 4 = Enhancing Tumor
    # Exclude Label 2 to track solid tumor instead of swelling
    solid_mask = np.logical_or(cleaned_mask == 1, cleaned_mask == 4)
    solid_volume_mm3 = np.sum(solid_mask) * voxel_volume_mm3

    return true_foci_count, solid_volume_mm3, cleaned_mask


class GeneralizedDiceLoss(nn.Module):
    """
    Solves extreme class imbalance by weighting the spatial overlap
    inversely to class volume.
    """
    def __init__(self, epsilon=1e-6):
        super(GeneralizedDiceLoss, self).__init__()
        self.epsilon = epsilon

    def forward(self, predictions, targets):
        class_volumes = torch.sum(targets, dim=(0, 2, 3, 4))
        weights = 1.0 / ((class_volumes ** 2) + self.epsilon)

        intersection = torch.sum(predictions * targets, dim=(0, 2, 3, 4))
        union = torch.sum(predictions + targets, dim=(0, 2, 3, 4))

        dice_score = 2.0 * torch.sum(weights * intersection) / torch.sum(weights * union + self.epsilon)
        return 1.0 - dice_score


print("Utility functions and GeneralizedDiceLoss successfully defined.")

Utility functions and GeneralizedDiceLoss successfully defined.


---

### Cell 3: Virtual Cataloging & Micro-Batch Pipeline

In [3]:
# Setup directory paths
brats_root = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
raw_temp_dir = "/kaggle/working/raw_temp"
processed_temp_dir = "/kaggle/working/processed_temp"
final_archive_path = "/kaggle/working/brats2021_processed.tar"

# Remove existing output archive if re-running cell
if os.path.exists(final_archive_path):
    os.remove(final_archive_path)

# Known outliers / duplicate scans identified during EDA
invalid_cases = {
    "BraTS2021_00495",  # Duplicate scan
    "BraTS2021_00621",  # Duplicate scan
    "BraTS2021_01163"   # Extreme brightness anomaly
}

BATCH_SIZE = 50  # Process 50 patients per batch to remain safely under Kaggle's disk limit

# --- Step 1: Scan archives to create a global virtual catalog (uses negligible memory/disk) ---
print("Step 1: Building global virtual catalog of patients...")
tar_files = glob.glob(f"{brats_root}/*.tar")
global_patients = defaultdict(lambda: defaultdict(list))

for tar_path in tar_files:
    with tarfile.open(tar_path, "r") as tf:
        for member_name in tf.getnames():
            parts = member_name.split('/')
            if len(parts) > 0 and parts[0].startswith("BraTS2021_"):
                patient_id = parts[0]
                if patient_id not in invalid_cases:
                    global_patients[patient_id][tar_path].append(member_name)

all_patient_ids = sorted(list(global_patients.keys()))
total_patients = len(all_patient_ids)
print(f"Catalog complete! Found {total_patients} valid patient directories across all .tar files.")

# --- Step 2: Micro-Batch Extraction, Normalization, Archiving, and Cleanup ---
print("\nStep 2: Processing in micro-batches to prevent disk space exhaustion...")

total_batches = (total_patients + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(total_batches):
    start_i = batch_idx * BATCH_SIZE
    end_i = min(start_i + BATCH_SIZE, total_patients)
    batch_ids = all_patient_ids[start_i:end_i]

    print(f"\n--- Batch {batch_idx + 1}/{total_batches} (Patients {start_i + 1} to {end_i} of {total_patients}) ---")

    # Re-create clean working directories for this batch
    os.makedirs(raw_temp_dir, exist_ok=True)
    os.makedirs(processed_temp_dir, exist_ok=True)

    # A. Extract only current batch files
    batch_extraction_plan = defaultdict(list)
    for pid in batch_ids:
        for tar_path, member_names in global_patients[pid].items():
            batch_extraction_plan[tar_path].extend(member_names)

    for tar_path, member_names in batch_extraction_plan.items():
        with tarfile.open(tar_path, "r") as tf:
            members_to_extract = [tf.getmember(name) for name in member_names]
            tf.extractall(path=raw_temp_dir, members=members_to_extract)

    # B. Process batch MRI volumes and copy segmentation masks
    for pid in tqdm(batch_ids, desc=f"Preprocessing Batch {batch_idx + 1}"):
        in_folder = os.path.join(raw_temp_dir, pid)
        out_folder = os.path.join(processed_temp_dir, pid)
        os.makedirs(out_folder, exist_ok=True)

        if not os.path.exists(in_folder):
            continue

        for file_name in os.listdir(in_folder):
            in_file = os.path.join(in_folder, file_name)
            out_file = os.path.join(out_folder, file_name)

            if file_name.endswith(".nii.gz") and not file_name.endswith("_seg.nii.gz"):
                # Load MRI image, apply clipping & Z-score normalization
                nii_img = nib.load(in_file)
                volume_data = nii_img.get_fdata()
                processed_data = preprocess_mri_volume(volume_data)

                # Save normalized array to processed output directory
                processed_img = nib.Nifti1Image(processed_data, nii_img.affine, nii_img.header)
                nib.save(processed_img, out_file)
            elif file_name.endswith("_seg.nii.gz"):
                # Pass segmentation mask directly without modifying intensity values
                shutil.copy2(in_file, out_file)

    # C. Append batch to uncompressed .tar archive
    with tarfile.open(final_archive_path, "a") as archive:
        for item in os.listdir(processed_temp_dir):
            archive.add(os.path.join(processed_temp_dir, item),
                        arcname=os.path.join("brats2021", item))

    # D. Delete uncompressed raw and processed files to free up disk immediately
    shutil.rmtree(raw_temp_dir)
    shutil.rmtree(processed_temp_dir)

print("\n*** ALL PATIENTS PROCESSED AND ARCHIVED SUCCESSFULLY! ***")

Step 1: Building global virtual catalog of patients...
Catalog complete! Found 0 valid patient directories across all .tar files.

Step 2: Processing in micro-batches to prevent disk space exhaustion...

*** ALL PATIENTS PROCESSED AND ARCHIVED SUCCESSFULLY! ***


---

### Cell 4: Verification & Disk Usage Summary

In [4]:
if os.path.exists(final_archive_path):
    size_gb = os.path.getsize(final_archive_path) / 1e9
    print("Validation successful!")
    print(f"Archive path: {final_archive_path}")
    print(f"Total size on disk: {size_gb:.2f} GB")

    # Inspect internal archive structure
    with tarfile.open(final_archive_path, "r") as archive:
        members = archive.getnames()
        sample_dirs = set(m.split('/')[1] for m in members if '/' in m and m.split('/')[1].startswith("BraTS2021_"))
        print(f"Total archived files: {len(members)}")
        print(f"Total clean patient directories stored: {len(sample_dirs)}")
else:
    print("Error: Archive file was not found.")

Error: Archive file was not found.
